# 01 — Load Collection A and merge by proteomics platform

This notebook intentionally mirrors only the **Collection A loading step** from `05. validation/notebooks/1. process_collection_a.ipynb`:

1. discover the 20 cohort folders;
2. assign cohorts to **TMT**, **iTRAQ**, or **LabelFree**;
3. load `protein_matrix_*.txt` and `RNA_matrix_*.txt`;
4. align matched RNA/protein samples within each cohort;
5. map Ensembl identifiers to gene symbols and collapse duplicates;
6. filter features and samples by non-missing fraction;
7. save one RNA matrix, one protein matrix, and metadata table per platform.

No normalization or train/test matching is performed here. Those operations are isolated in notebook 02 so each transfer scenario learns parameters from its training datasets only.

In [1]:
from pathlib import Path
import sys
import joblib
import pandas as pd

NOTEBOOK_DIR = Path.cwd().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent if NOTEBOOK_DIR.name == "notebooks" else NOTEBOOK_DIR
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))

from utils import build_collection_a_platforms, load_yaml

print("PROJECT_ROOT:", PROJECT_ROOT)

PROJECT_ROOT: C:\Users\hp\Desktop\CPPR\Cancer-Protein-RNA-research\07.multidataset_transfer


## Global configuration

In [2]:
# Edit only this cell.
COLLECTION_DIR = PROJECT_ROOT / "data" / "Collection_A"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "platforms"
GROUPS_CONFIG = PROJECT_ROOT / "configs" / "collection_a_groups.yaml"

# Recommended: provide a local TSV with columns such as
# ensembl_gene_id and gene_symbol. This is reproducible and avoids network dependence.
MAPPING_PATH = PROJECT_ROOT / "data" / "reference" / "ensembl_to_symbol.tsv"
if not MAPPING_PATH.exists():
    MAPPING_PATH = None

# Set True only once if no local mapping exists and this machine has internet access.
# The queried mapping is cached under data/reference/ensembl_to_symbol_cache.tsv.
ALLOW_MYGENE_QUERY = False

FEATURE_NONMISSING_FRACTION = 0.50
SAMPLE_NONMISSING_FRACTION = 0.50
MAP_PROTEIN_IDS = True
FORCE_REBUILD = False

COHORT_GROUPS = load_yaml(GROUPS_CONFIG)["collection_a"]
COHORT_GROUPS

{'TMT': ['colon_2019',
  'renal_2019',
  'Endomet_val',
  'breast2020',
  'pedbrain_2020',
  'lung_2020_smoke',
  'lung_val',
  'lung2021',
  'headneck_2021',
  'pancrea2021',
  'brain_2021',
  'liver_val'],
 'iTRAQ': ['breast_2016', 'ovary_2016', 'stomach_2019'],
 'LabelFree': ['brain', 'colon', 'prostate', 'liver', 'lung']}

## Build TMT, iTRAQ, and LabelFree matrices

In [3]:
cohort_summary, platform_summary = build_collection_a_platforms(
    collection_dir=COLLECTION_DIR,
    output_dir=OUTPUT_DIR,
    cohort_groups=COHORT_GROUPS,
    mapping_path=MAPPING_PATH,
    allow_mygene_query=ALLOW_MYGENE_QUERY,
    feature_nonmissing_fraction=FEATURE_NONMISSING_FRACTION,
    sample_nonmissing_fraction=SAMPLE_NONMISSING_FRACTION,
    map_protein_ids=MAP_PROTEIN_IDS,
    force=FORCE_REBUILD,
)

display(cohort_summary)
display(platform_summary)

RuntimeError: Collection A contains Ensembl identifiers, but no mapping is available. Set MAPPING_PATH to a local TSV or run with ALLOW_MYGENE_QUERY=1 once.

## Validate saved platform bundles

In [ ]:
validation_rows = []
for platform in ["TMT", "iTRAQ", "LabelFree"]:
    platform_dir = OUTPUT_DIR / platform
    X = joblib.load(platform_dir / "transcriptomics.joblib")
    Y = joblib.load(platform_dir / "proteomics.joblib")
    meta = pd.read_csv(platform_dir / "metadata.tsv", sep="\t", index_col=0)
    common = X.index.intersection(Y.index).intersection(meta.index)
    validation_rows.append({
        "platform": platform,
        "RNA shape": X.shape,
        "protein shape": Y.shape,
        "metadata rows": len(meta),
        "three-way sample overlap": len(common),
        "RNA missing %": 100 * X.isna().mean().mean(),
        "protein missing %": 100 * Y.isna().mean().mean(),
    })

validation = pd.DataFrame(validation_rows)
display(validation)
assert (validation["three-way sample overlap"] == validation["metadata rows"]).all()
print("Collection A loading completed successfully.")